In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb
import optuna
import lightgbm as lgb

In [ ]:
df = pd.read_csv('../data/002_processed_data/xy_df_hopv.csv')
print(df.shape)
print(df['stability'].shape)
print(df['stability'].isna().sum())
df = df.dropna(subset=['stability']) 
print(df['stability'].isna().sum())
X = df[[column for column in df if column not in ['smiles', 'stability', 'homo', 'lumo']]]
feature_columns = X.columns.tolist()
y = df['stability']
le = LabelEncoder()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth = 5,
    eval_metric='mlogloss',
    random_state=42
    )
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

gbc = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
accuracy_xgb= cross_val_score(xgb_model, X_train, y_train,cv=cv, scoring='accuracy').mean()
accuracy_rfc = cross_val_score(rfc, X_train, y_train, cv=cv, scoring='accuracy').mean()
f1_xgb = cross_val_score(xgb_model, X_train, y_train,cv=cv, scoring='f1_macro').mean()
f1_rfc= cross_val_score(rfc, X_train, y_train,cv=cv, scoring='f1_macro').mean()

print(f'Accuracy XGB: {accuracy_xgb} Accuracy RFC: {accuracy_rfc}')
print(f'F1 XGB: {f1_xgb} F1 RFC: {f1_rfc}')

In [ ]:
best_params = {'n_estimators': 82,
        'max_depth': 4,
            'learning_rate': 0.047732412372341396,
            'subsample': 0.9859797724837484,
                'colsample_bytree': 0.927320180766563,
                'min_child_weight': 4,
                    'gamma': 1.3644788423904113,
                    'reg_alpha': 4.538324332642517e-07,
                        'reg_lambda': 0.005193227796473285}

best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_pred=y_pred, y_true=y_test)
f1 = f1_score(y_pred=y_pred, y_true=y_test, average='macro')
print(accuracy, f1)

In [ ]:
best_model = xgb.XGBClassifier(**best_params)
y = le.transform(y)
best_model.fit(X, y)

In [ ]:
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
data = pd.read_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv')
smiles = data['smiles']
smiles

In [ ]:
descriptor_names = [x[0] for x in Descriptors._descList]


In [ ]:
def compute_descriptors(smiles_str):
    """
    WRITE A DESCRIPTION

    Parameters
    ----------
    smiles_str : str
        The SMILES string representation of a molecule.

    Returns
    -------
    tuple
        A tuple containing the values of all available molecular descriptors
        (217 descriptors in total, from RDKit descriptor list).
    """
    # YOUR CODE HERE
    mol = Chem.MolFromSmiles(smiles_str)
    features = tuple(getattr(Descriptors, name)(mol) for name in descriptor_names)
    return features


In [ ]:
descriptors_df = pd.DataFrame(smiles.apply(compute_descriptors).tolist(), columns=descriptor_names)
descriptors_features = descriptors_df.columns.tolist()


In [ ]:
y_pred = best_model.predict(descriptors_df)
y_pred = le.inverse_transform(y_pred)
submission_df = pd.DataFrame({
    'smiles': smiles,
    'stability': y_pred
})
pd.DataFrame(submission_df).to_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv', index=False)


In [ ]:
submission = pd.read_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv')